# Bài 1 · Vector và độ tương đồng cosine · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Dữ liệu: `data/raw/movielens.zip`, bản lưu MovieLens latest-small (GroupLens; Harper, Konstan 2015), kiểm SHA-256 ở ô đầu.
Mã cho sẵn; việc của bạn là chạy từng ô, đối chiếu với slide và trả lời câu hỏi **in đậm**. Ô có dấu ✅ là ô tự kiểm.

Chặng 4 chỉ dùng **nhóm bài giảng** (300 người, đã xuất hiện trên slide). Nhóm phát triển và nhóm kiểm tra cuối để dành cho project, notebook này không đọc tới.

In [1]:
from pathlib import Path
import hashlib, json, math, zipfile
from fractions import Fraction
import numpy as np, pandas as pd, scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw' / 'movielens.zip'
assert hashlib.sha256(RAW.read_bytes()).hexdigest() == '12b11d77fdb2c221daa631e5724104f51ecab1298c0e5f535d56e9ceff02ef8b', 'sai bản lưu dữ liệu'
z = zipfile.ZipFile(RAW)
mv = pd.read_csv(z.open('movies.csv')); rt = pd.read_csv(z.open('ratings.csv'))
def kiem(ten, dk):
    assert dk, '❌ ' + ten
    print('✅', ten)
print(len(mv), 'phim ·', len(rt), 'lượt đánh giá ·', rt.userId.nunique(), 'người xem')

9742 phim · 100836 lượt đánh giá · 610 người xem


## Chặng 1 · Phim thành vector

In [2]:
labels = [set(g.split('|')) - {'(no genres listed)'} for g in mv.genres]
AX = sorted({a for s in labels for a in s})            # 19 trục, thứ tự cố định theo bảng chữ cái
X = np.array([[a in s for a in AX] for s in labels], dtype=float)
AX3 = ['Action', 'Comedy', 'Sci-Fi']; X3 = X[:, [AX.index(a) for a in AX3]]
idx = {m: i for i, m in enumerate(mv.movieId)}
print(len(AX), 'trục:', AX)
print('The Matrix, 19 trục:', X[idx[2571]].astype(int))
print('The Matrix, 3 trục :', X3[idx[2571]].astype(int))
corners = pd.Series([''.join(map(str, r)) for r in X3.astype(int)]).value_counts()
print(corners)

19 trục: ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
The Matrix, 19 trục: [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0]
The Matrix, 3 trục : [1 0 1]
000    4196
010    3189
100     999
101     400
001     391
110     378
011     138
111      51
Name: count, dtype: int64


In [3]:
kiem('19 trục, có IMAX', len(AX) == 19 and 'IMAX' in AX)
kiem('4196 phim là vector 0 trên ba trục (slide 10)', corners['000'] == 4196)
kiem('34 phim không có nhãn nào', int((X.sum(1) == 0).sum()) == 34)
kiem('chỉ có 8 đỉnh', len(corners) == 8)

✅ 19 trục, có IMAX
✅ 4196 phim là vector 0 trên ba trục (slide 10)
✅ 34 phim không có nhãn nào
✅ chỉ có 8 đỉnh


**Câu hỏi.** Vì sao Star Wars IV và The Matrix khác nhau mà cùng một vector trên ba trục? Thêm trục nào thì tách được?

## Chặng 2 · Tích vô hướng, độ dài, cosine

In [4]:
def cosine_safe(u, v):
    u = np.asarray(u, float); v = np.asarray(v, float)
    if u.shape != v.shape or u.ndim != 1 or u.size == 0:
        raise ValueError('hai vector phải cùng số chiều và không rỗng')
    if not (np.isfinite(u).all() and np.isfinite(v).all()):
        raise ValueError('vector chứa NaN hoặc vô cực')
    nu, nv = math.sqrt(u @ u), math.sqrt(v @ v)
    if nu == 0 or nv == 0:
        return None            # không xác định
    return float(u @ v) / (nu * nv)

u, v = np.array([3, 1]), np.array([1, 2])
print('u·v =', u @ v, '· ‖(3;4)‖ =', np.linalg.norm([3, 4]))
print('cos((1;2),(2;1)) =', cosine_safe([1, 2], [2, 1]), '· nhân (1;2) với 3:', cosine_safe([3, 6], [2, 1]), '· với −1:', cosine_safe([-1, -2], [2, 1]))
print('u=(1;0): Euclid tới (1;1) =', math.dist([1, 0], [1, 1]), ', tới (3;0) =', math.dist([1, 0], [3, 0]))

u·v = 5 · ‖(3;4)‖ = 5.0
cos((1;2),(2;1)) = 0.7999999999999998 · nhân (1;2) với 3: 0.7999999999999999 · với −1: -0.7999999999999998
u=(1;0): Euclid tới (1;1) = 1.0 , tới (3;0) = 2.0


In [5]:
kiem('‖(3; 4)‖ = 5', np.linalg.norm([3, 4]) == 5)
kiem('cos((1;2),(2;1)) = 0,8 và không đổi khi nhân 3', abs(cosine_safe([1, 2], [2, 1]) - 0.8) < 1e-12 and abs(cosine_safe([3, 6], [2, 1]) - 0.8) < 1e-12)
kiem('nhân −1 thì đổi dấu', abs(cosine_safe([-1, -2], [2, 1]) + 0.8) < 1e-12)
kiem('vector 0 cho None', cosine_safe([1, 0, 1], [0, 0, 0]) is None)
for bad in ([1, 2], [np.nan, 0, 1]):
    try:
        cosine_safe([1, 0, 1], bad); ok = False
    except ValueError:
        ok = True
    kiem(f'chặn đầu vào hỏng {bad}', ok)

✅ ‖(3; 4)‖ = 5
✅ cos((1;2),(2;1)) = 0,8 và không đổi khi nhân 3
✅ nhân −1 thì đổi dấu
✅ vector 0 cho None
✅ chặn đầu vào hỏng [1, 2]
✅ chặn đầu vào hỏng [nan, 0, 1]


**Câu hỏi.** Tính tay cos((1; 2; 2), (2; 1; 2)) rồi kiểm bằng `cosine_safe`.

## Chặng 3 · Tìm phim giống The Matrix trong cả kho

In [6]:
pop = rt.movieId.map(idx).value_counts().reindex(range(len(mv)), fill_value=0).values   # số lượt đánh giá, dùng phá đồng hạng
def top_k(q_movie, k=10, M=X, rule='pop'):
    norm = np.linalg.norm(M, axis=1)
    Mn = np.divide(M, norm[:, None], out=np.zeros_like(M), where=norm[:, None] > 0)
    q = idx[q_movie]
    if norm[q] == 0:
        return None, None, None          # truy vấn là vector 0: cosine không xác định
    s = np.round(Mn @ Mn[q], 12)
    ok = norm > 0; ok[q] = False
    ids = mv.movieId.values
    order = np.lexsort((ids, -pop, -s)) if rule == 'pop' else np.lexsort((ids, -s))
    order = [i for i in order if ok[i]]
    return order[:k], s, ok

top, s, ok = top_k(2571, 10)
print(pd.DataFrame({'phim': mv.title.values[top], 'cosine': s[top], 'lượt': pop[top]}))
vals, cnt = np.unique(s[ok], return_counts=True)
print(len(vals), 'giá trị khác nhau; số phim cosine 1:', cnt[vals == 1])

                                phim  cosine  lượt
0             Terminator, The (1984)     1.0   131
1                Blade Runner (1982)     1.0   124
2                    Predator (1987)     1.0    61
3             Johnny Mnemonic (1995)     1.0    53
4       X-Men: The Last Stand (2006)     1.0    52
5                 Equilibrium (2002)     1.0    44
6                    Godzilla (1998)     1.0    33
7                 Island, The (2005)     1.0    31
8    X-Men Origins: Wolverine (2009)     1.0    26
9  Chronicles of Riddick, The (2004)     1.0    23
20 giá trị khác nhau; số phim cosine 1: [42]


In [7]:
kiem('9707 phim được so', int(ok.sum()) == 9707)
kiem('20 giá trị cosine khác nhau (slide 44)', len(vals) == 20)
kiem('42 phim cosine đúng bằng 1 (slide 45)', int(cnt[vals == 1][0]) == 42)
kiem('top 10 theo quy tắc “nhiều lượt trước” khớp slide', [int(mv.movieId[i]) for i in top] == [1240, 541, 3527, 172, 45499, 5903, 1882, 34319, 68319, 8371])
top_id, _, _ = top_k(2571, 10, rule='id')
kiem('top 10 theo “movieId nhỏ trước” khớp slide', [int(mv.movieId[i]) for i in top_id] == [66, 76, 172, 338, 379, 541, 692, 748, 1240, 1882])
# công thức đếm |A∩B|/√(|A||B|) bằng phân số, cho mọi phim
A = labels[idx[2571]]
err = max(abs(s[i] - math.sqrt(Fraction(len(A & labels[i]) ** 2, len(A) * len(labels[i])))) for i in np.flatnonzero(ok))
kiem('công thức đếm khớp phép nhân ma trận cho cả kho', err < 1e-11)
sk = cosine_similarity(X[[idx[m] for m in (260, 1580, 480, 1214, 1)]], X[[idx[2571]]]).ravel()
kiem('khớp scikit-learn trên năm phim', np.allclose(sk, [s[idx[m]] for m in (260, 1580, 480, 1214, 1)]))
kiem('scikit-learn trả 0 cho vector 0 (quy ước), hàm của bài trả None', cosine_similarity([[1, 0, 1]], [[0, 0, 0]])[0, 0] == 0 and cosine_safe([1, 0, 1], [0, 0, 0]) is None)

✅ 9707 phim được so
✅ 20 giá trị cosine khác nhau (slide 44)
✅ 42 phim cosine đúng bằng 1 (slide 45)
✅ top 10 theo quy tắc “nhiều lượt trước” khớp slide
✅ top 10 theo “movieId nhỏ trước” khớp slide
✅ công thức đếm khớp phép nhân ma trận cho cả kho
✅ khớp scikit-learn trên năm phim
✅ scikit-learn trả 0 cho vector 0 (quy ước), hàm của bài trả None


**Câu hỏi.** Đổi `rule='id'`. Hai top 10 chung bao nhiêu phim? Quy tắc nào bạn sẽ chọn, và vì sao phải chọn **trước** khi xem kết quả?

## Chặng 4 · Cùng phép đo, đổi biểu diễn (chỉ nhóm bài giảng)

Giao thức ghi trước (`results/protocol_before_eval.json`): giấu phim thích (≥ 4) được chấm sau cùng; chỉ dùng lượt chấm trước thời điểm đó; ứng viên là phim chưa chấm; ba cách xếp hạng; hit@20.

In [8]:
U = json.loads((ROOT / 'data' / 'project_users_bai01.json').read_text())['lecture']       # chỉ nhóm bài giảng
rt['i'] = rt.movieId.map(idx)
Xn = X / np.maximum(np.linalg.norm(X, axis=1), 1e-300)[:, None] * (np.linalg.norm(X, axis=1) > 0)[:, None]
nU, nM = rt.userId.max() + 1, len(mv)
def mot_nguoi(u):
    g = rt[rt.userId == u].sort_values(['timestamp', 'movieId']); l = g[g.rating >= 4]
    T = int(l.timestamp.iloc[-1]); tgt = int(l.i.iloc[-1])
    hist = g[g.timestamp < T]; liked = hist[hist.rating >= 4].i.values
    before = rt[rt.timestamp < T]
    popT = np.bincount(before.i.values, minlength=nM).astype(float)
    cand = np.ones(nM, bool); cand[hist.i.values] = False
    p = Xn[liked].sum(0); s_g = Xn @ p / np.linalg.norm(p)
    b = before[before.rating >= 4]
    B = sp.csr_matrix((np.ones(len(b)), (b.i.values, b.userId.values)), shape=(nM, nU))
    nb = np.sqrt(np.asarray(B.multiply(B).sum(1)).ravel())
    Sm = (B[liked] @ B.T).toarray(); den = nb[liked][:, None] * nb[None, :]
    s_b = np.divide(Sm, den, out=np.zeros_like(Sm), where=den > 0).sum(0)
    out = {}
    for k, sc in (('genre', s_g), ('behaviour', s_b), ('popular', popT)):
        s2 = np.where(cand, np.round(sc, 12), -np.inf)
        out[k] = tgt in np.lexsort((np.arange(nM), -popT, -s2))[:20]
    return out
hits = [mot_nguoi(u) for u in U]
tong = {k: sum(h[k] for h in hits) for k in ('genre', 'popular', 'behaviour')}
chi_b = sum(h['behaviour'] and not h['popular'] for h in hits); chi_p = sum(h['popular'] and not h['behaviour'] for h in hits)
print(tong, '· chỉ hành vi trúng', chi_b, '· chỉ phổ biến trúng', chi_p)

{'genre': 5, 'popular': 29, 'behaviour': 35} · chỉ hành vi trúng 21 · chỉ phổ biến trúng 15


In [9]:
kiem('hit@20 khớp slide 56: nhãn 5, phổ biến 29, hành vi 35', tong == {'genre': 5, 'popular': 29, 'behaviour': 35})
kiem('so theo cặp khớp slide: 21 so với 15', (chi_b, chi_p) == (21, 15))
from scipy.stats import binomtest
print('kiểm định dấu hai phía p =', round(binomtest(chi_b, chi_b + chi_p, 0.5).pvalue, 3))

✅ hit@20 khớp slide 56: nhãn 5, phổ biến 29, hành vi 35
✅ so theo cặp khớp slide: 21 so với 15
kiểm định dấu hai phía p = 0.405


**Câu hỏi.**
1. Vì sao độ phổ biến cũng phải đếm **chỉ** từ lượt chấm trước thời điểm giấu?
2. Hành vi trúng nhiều hơn mốc phổ biến. Kết luận “hành vi tốt hơn” có đúng mức không? Dùng số người chỉ một cách trúng để trả lời.
3. Thời điểm trong MovieLens là lúc *nhập* điểm. Điều đó thay đổi cách đọc “phim thích tiếp theo” thế nào?